# Perpetrator classifier — versión mejorada para screening

Script backend para construir y evaluar el clasificador de perpetrador.<br>
Junta todas las celdas de código en una.<br>
Construye la tabla con datos, el conjunto de entrenamiento y validación.<br>
Entrena el modelo DNN, lo evalúa y muestra los resultados de la evaluación como resultado final.<br>


## Mejoras introducidas en esta versión

Esta versión está ajustada al objetivo del estudio: **identificar el máximo número posible de adolescentes perpetradores**, priorizando el recall/sensibilidad de la clase positiva. Para ello, el modelo ya no se evalúa únicamente con el umbral estándar de 0.50, sino que prueba varios umbrales más bajos para favorecer el cribado. Además, se incrementa el peso de la clase perpetrador durante el entrenamiento, se fijan semillas para reducir variabilidad entre ejecuciones y se guarda explícitamente el **mejor modelo real**, junto con su threshold, métricas y predicciones. También se evita un problema importante de la versión anterior: el CSV final podía estar guardando las predicciones del último modelo entrenado, no necesariamente las del mejor modelo encontrado.

La selección del mejor modelo sigue priorizando no dejarnos perpetradores fuera, pero incorpora mínimos de especificidad y precisión para evitar soluciones degeneradas que clasifiquen casi todos los casos como perpetradores. Por tanto, el resultado debe interpretarse como un **modelo de screening sensible**, útil para detectar posibles casos y priorizar revisión posterior, no como una clasificación diagnóstica definitiva.


### Pre-condición
Tu gráfica puede ayudarte con el cálculo, comprúebalo ejecutando el código de aqui abajo👇

In [ ]:
#%pip install torch
import torch
torch.cuda.is_available()

### Ejecución
El siguiente código realiza el proceso de clasificación explicado al comienzo.

In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import joblib
import os
import re
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import recall_score
from collections import Counter
from sklearn.metrics import classification_report

import os
import itertools
import json
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, Callback
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

def perform_pca(df: pd.DataFrame,
                save_scaler_path: str = "models/scaler_minmax.pkl",
                save_pca_path: str = "models/modelo_pca.pkl",
                save_means_path: str = "models/medias_escalado.csv"
               ):
    """
    Realiza un PCA sobre el DataFrame de entrada y devuelve:
      1) df_pca: DataFrame con las componentes principales (PC1, PC2, …).
      2) df_var: DataFrame con las varianzas explicada y acumulada por componente.
      3) scaler: MinMaxScaler ya entrenado.
      4) pca: PCA ya entrenado.
      5) means: Series con la media de cada variable tras escalado.
    Además guarda automáticamente el scaler, el modelo PCA y el vector de medias en rutas predeterminadas.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame numérico de entrada (n muestras × p variables).
    save_scaler_path : str
        Ruta por defecto donde volcar el scaler (pickle).
    save_pca_path : str
        Ruta por defecto donde volcar el modelo PCA (pickle).
    save_means_path : str
        Ruta por defecto donde guardar las medias de cada variable tras escalado (CSV).

    Retorna
    -------
    df_pca : pd.DataFrame
        Transformación PCA (n muestras × p componentes), columnas PC1…PCp.
    df_var : pd.DataFrame
        Tabla con columnas:
          - PC: nombre de la componente ("PC1", "PC2", …)
          - explained variance: varianza explicada por cada componente
          - cumulative variance: varianza acumulada hasta esa componente
    scaler : MinMaxScaler
        Objeto scaler ajustado (fit).
    pca : PCA
        Objeto PCA ajustado (fit).
    means : pd.Series
        Media de cada variable tras el escalado (para centrar nuevos datos).
    """
    # Crear carpetas si no existen

    os.makedirs(os.path.dirname(save_scaler_path), exist_ok=True)
    os.makedirs(os.path.dirname(save_pca_path), exist_ok=True)
    os.makedirs(os.path.dirname(save_means_path), exist_ok=True)

    # 1) Escalado al rango [0,1]
    scaler = MinMaxScaler(feature_range=(0, 1))
    X_scaled = scaler.fit_transform(df.values)

    # 2) Cálculo de medias para centrar
    means_array = X_scaled.mean(axis=0)
    means = pd.Series(means_array, index=df.columns, name='mean')

    # 3) Centramos cada variable usando las medias calculadas
    X_centered = X_scaled - means_array

    # 4) Ajustar y transformar PCA
    pca = PCA(n_components=df.shape[1])
    comps = pca.fit_transform(X_centered)

    # 5) DataFrame de componentes principales
    cols = [f"PC{i+1}" for i in range(df.shape[1])]
    df_pca = pd.DataFrame(comps, index=df.index, columns=cols)

    # 6) Varianzas explicada y acumulada
    var_ratio = pca.explained_variance_ratio_
    cum_var  = var_ratio.cumsum()
    df_var = pd.DataFrame({
        "PC": cols,
        "explained variance": var_ratio,
        "cumulative variance": cum_var
    })

    # 7) Guardar a disco en rutas por defecto
    joblib.dump(scaler, save_scaler_path)
    joblib.dump(pca, save_pca_path)
    means.to_csv(save_means_path, header=True)

    return df_pca, df_var, scaler, pca, means

def plot_pca_variance(df_var):
    """
    Genera un barplot de la varianza explicada (barras amarillas) y un lineplot
    de la varianza acumulada (línea azul), marcando con un punto rojo donde
    la varianza acumulada supera el 0.95.

    Parámetros
    ----------
    df_var : pandas.DataFrame
        DataFrame con columnas:
          - "PC": nombres de componentes ("PC1", "PC2", …)
          - "explained variance": varianza explicada por componente
          - "cumulative variance": varianza acumulada
    """
    # Preparar datos
    x = range(len(df_var))
    explained = df_var['explained variance']
    cumulative = df_var['cumulative variance']

    # Crear la figura
    plt.figure(figsize=(10, 6))

    # Barplot de varianza explicada
    plt.bar(x, explained, color='orange', label='Explained Variance')

    # Lineplot de varianza acumulada
    plt.plot(x, cumulative, color='blue', marker='o', label='Cumulative Variance')

    # Punto rojo donde cumulative > 0.95
    above_idx = df_var.index[cumulative > 0.95]
    if not above_idx.empty:
        idx0 = above_idx[0]
        plt.scatter(idx0, cumulative[idx0], color='red', zorder=5, label='Cumulative > 0.95')
        plt.axvline(x=idx0, color='red', linestyle='--')

    # Configuración de ejes y leyenda
    plt.xticks(x, df_var['PC'], rotation=90)
    plt.xlabel('Principal Components')
    plt.ylabel('Variance')
    plt.title('Explained and Cumulative Variance by PCA Components')
    plt.legend()
    plt.tight_layout()
    plt.show()


def select_PCA_df(df_var: pd.DataFrame, df_PCA: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """
    Selecciona las primeras i componentes principales de df_PCA donde i es el número de componente
    tal que el primer valor de 'cumulative variance' en df_var supera el umbral dado.

    Args:
        df_var (pd.DataFrame): DataFrame con columnas 'PC' (etiquetas 'PC1', 'PC2', ...) y
                               'cumulative variance' (float entre 0 y 1).
        df_PCA (pd.DataFrame): DataFrame resultante de una PCA, con tantas columnas como componentes.
        threshold (float): Umbral de varianza acumulada entre 0 y 1.

    Returns:
        pd.DataFrame: Subset de df_PCA con las primeras i columnas.
    """
    # Validación de umbral
    if not (0 <= threshold <= 1):
        raise ValueError("El umbral debe estar entre 0 y 1")

    # Buscar el primer índice que supere el threshold
    mask = df_var['cumulative variance'] > threshold
    if not mask.any():
        raise ValueError("Ningún valor de varianza acumulada supera el umbral dado")

    idx = mask.idxmax()  # índice del primer True
    pc_label = df_var.loc[idx, 'PC']  # p.ej. 'PC3'

    # Extraer i del string 'PCi'
    match = re.match(r'PC(\d+)', pc_label)
    if not match:
        raise ValueError(f"Formato inesperado en etiqueta PC: {pc_label}")
    i = int(match.group(1))

    # Seleccionar las primeras i columnas de df_PCA
    return df_PCA.iloc[:, :i]

def NN_perpetrator_classifier(
    df_pca_reduced,
    df_merged_perpetrador_target,
    output_dir='./content/perpetrator',
    batch_size=128,
    grid_params=None,
    tpu_address=None,
    seed=42,
    recall_priority_weight=1.25,
    thresholds=None,
    min_specificity=0.25,
    min_precision=0.20,
    target_recall=0.90,
    use_mixed_precision=False
):
    """
    Entrena un clasificador neuronal orientado a SCREENING de perpetradores.

    Objetivo principal:
    - Priorizar recall/sensibilidad de la clase 1 (PERPETRADOR), es decir, minimizar falsos negativos.
    - No seleccionar modelos degenerados que clasifican casi todo como positivo sin un mínimo de especificidad/precisión.

    Mejoras frente a la versión anterior:
    - Fija semillas para reducir variabilidad entre ejecuciones.
    - Divide por posiciones y conserva el índice original para evitar errores si el índice del DataFrame no es 0..n-1.
    - Evalúa varios thresholds, no solo 0.50.
    - Selecciona y guarda realmente el mejor modelo + threshold, no el último modelo entrenado del grid.
    - Guarda predicciones con probabilidad, threshold usado y métricas completas para Smart AND.
    """

    import random
    from sklearn.metrics import (
        classification_report,
        confusion_matrix,
        precision_score,
        recall_score,
        f1_score,
        accuracy_score,
        balanced_accuracy_score
    )

    os.makedirs(output_dir, exist_ok=True)

    # 1. Reproducibilidad
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
        print("Determinismo TensorFlow activado.")
    except Exception as exc:
        print("No se pudo activar determinismo estricto en TensorFlow:", exc)

    # 2. Estrategia: TPU si existe; si no, CPU/GPU normal
    using_tpu = False
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver(tpu=tpu_address)
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
        using_tpu = True
        print("👾 TPU inicializado:", resolver.master())
    except (ValueError, tf.errors.NotFoundError):
        strategy = tf.distribute.get_strategy()
        print("⚠️ No se encontró TPU, usando", type(strategy).__name__)

    # 3. Precisión numérica
    # Para reproducibilidad y evitar sorpresas, por defecto usamos float32.
    # Si quieres velocidad en A100/H100/L4, puedes llamar use_mixed_precision=True.
    if use_mixed_precision:
        policy_name = "mixed_bfloat16" if using_tpu else "mixed_float16"
    else:
        policy_name = "float32"
    mixed_precision.set_global_policy(policy_name)
    print("Política de precisión:", mixed_precision.global_policy())

    # 4. Preparar datos
    X = df_pca_reduced.values.astype("float32")
    y = df_merged_perpetrador_target.astype(int).values
    original_idx = df_pca_reduced.index.to_numpy()
    positions = np.arange(len(X))

    train_pos, val_pos = train_test_split(
        positions,
        test_size=0.25,
        random_state=seed,
        stratify=y
    )

    splits = {
        'train_pos': train_pos.tolist(),
        'val_pos': val_pos.tolist(),
        'train_original_idx': original_idx[train_pos].tolist(),
        'val_original_idx': original_idx[val_pos].tolist(),
        'seed': seed
    }
    with open(os.path.join(output_dir, 'splits_indices.json'), 'w') as f:
        json.dump(splits, f, indent=2)

    X_train, X_val = X[train_pos], X[val_pos]
    y_train, y_val = y[train_pos], y[val_pos]
    val_original_idx = original_idx[val_pos]

    print(f"Train n={len(y_train)} | Val n={len(y_val)}")
    print("Distribución train:", dict(zip(*np.unique(y_train, return_counts=True))))
    print("Distribución val:", dict(zip(*np.unique(y_val, return_counts=True))))

    # 5. Pesos de clase.
    # El peso positivo se incrementa para favorecer no dejarnos perpetradores.
    unique_classes, counts = np.unique(y_train, return_counts=True)
    class_weight = {int(cls): 1.0 for cls in unique_classes}
    if 0 in unique_classes and 1 in unique_classes:
        n0 = counts[unique_classes == 0].sum()
        n1 = counts[unique_classes == 1].sum()
        class_weight[0] = 1.0
        class_weight[1] = float((n0 / n1) * recall_priority_weight)

    print("class_weight:", class_weight)

    # 6. Dataset
    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(10000, seed=seed, reshuffle_each_iteration=True)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((X_val, y_val))
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    # 7. Callback: no paramos solo por diferencia train-val demasiado pronto.
    # EarlyStopping sobre val_recall mantiene el foco en sensibilidad.
    callbacks = [
        EarlyStopping(
            monitor='val_recall',
            mode='max',
            patience=12,
            restore_best_weights=True
        )
    ]

    # 8. Grid por defecto algo más contenido que el original para no explotar combinaciones.
    # Puedes ampliarlo si tienes A100/H100 y tiempo.
    if grid_params is None:
        grid_params = {
            'u1': [64, 32],
            'a1': ['relu', 'tanh'],
            'd1': [0.3, 0.2],
            'u2': [16, 8],
            'a2': ['relu', 'tanh'],
            'd2': [0.1, 0.0]
        }

    if thresholds is None:
        # Para screening, umbrales bajos suelen aumentar recall.
        thresholds = np.round(np.arange(0.15, 0.56, 0.05), 2).tolist()

    def build_model(u1, a1, d1, u2, a2, d2):
        model = models.Sequential([
            layers.Input(shape=(X_train.shape[1],)),
            layers.Dense(u1, activation=a1),
            layers.Dropout(d1),
            layers.Dense(u2, activation=a2),
            layers.Dropout(d2),
            layers.Dense(1, activation='sigmoid', dtype='float32'),
        ])
        model.compile(
            optimizer=tf.keras.optimizers.Adam(1e-3),
            loss='binary_crossentropy',
            metrics=[
                tf.keras.metrics.Recall(name='recall'),
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.BinaryAccuracy(name='accuracy')
            ]
        )
        return model

    results = []
    best = None

    with strategy.scope():
        for u1, a1, d1, u2, a2, d2 in itertools.product(
            grid_params['u1'], grid_params['a1'], grid_params['d1'],
            grid_params['u2'], grid_params['a2'], grid_params['d2']
        ):
            model = build_model(u1, a1, d1, u2, a2, d2)

            history = model.fit(
                train_ds,
                validation_data=val_ds,
                epochs=200,
                class_weight=class_weight,
                callbacks=callbacks,
                verbose=0
            )

            prob_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)

            for threshold in thresholds:
                pred_val = (prob_val >= threshold).astype(int)

                tn, fp, fn, tp = confusion_matrix(y_val, pred_val, labels=[0, 1]).ravel()

                recall1 = recall_score(y_val, pred_val, pos_label=1, zero_division=0)
                recall0 = recall_score(y_val, pred_val, pos_label=0, zero_division=0)  # specificity
                precision1 = precision_score(y_val, pred_val, pos_label=1, zero_division=0)
                f1_1 = f1_score(y_val, pred_val, pos_label=1, zero_division=0)
                accuracy = accuracy_score(y_val, pred_val)
                balanced_acc = balanced_accuracy_score(y_val, pred_val)

                # Selección orientada a screening:
                # 1) recall de perpetradores pesa muchísimo.
                # 2) penalizamos si la especificidad o precisión caen por debajo de mínimos.
                # 3) mantenemos algo de balanced accuracy para evitar modelos "todo positivo".
                penalty = 0.0
                if recall0 < min_specificity:
                    penalty += (min_specificity - recall0) * 0.50
                if precision1 < min_precision:
                    penalty += (min_precision - precision1) * 0.30

                screening_score = (
                    0.82 * recall1 +
                    0.12 * balanced_acc +
                    0.06 * precision1 -
                    penalty
                )

                row = {
                    'u1': u1, 'a1': a1, 'd1': d1,
                    'u2': u2, 'a2': a2, 'd2': d2,
                    'threshold': float(threshold),
                    'recall_0_specificity': float(recall0),
                    'recall_1_sensitivity': float(recall1),
                    'precision_1_ppv': float(precision1),
                    'f1_1': float(f1_1),
                    'accuracy': float(accuracy),
                    'balanced_accuracy': float(balanced_acc),
                    'screening_score': float(screening_score),
                    'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)
                }
                results.append(row)

                is_better = best is None or row['screening_score'] > best['row']['screening_score']
                # Si ambos superan target_recall, desempata con menos FN y mayor especificidad.
                if best is not None:
                    both_high_recall = (
                        row['recall_1_sensitivity'] >= target_recall and
                        best['row']['recall_1_sensitivity'] >= target_recall
                    )
                    if both_high_recall:
                        is_better = (
                            row['FN'] < best['row']['FN'] or
                            (row['FN'] == best['row']['FN'] and row['recall_0_specificity'] > best['row']['recall_0_specificity']) or
                            (row['FN'] == best['row']['FN'] and row['recall_0_specificity'] == best['row']['recall_0_specificity'] and row['precision_1_ppv'] > best['row']['precision_1_ppv'])
                        )

                if is_better:
                    best = {
                        'row': row.copy(),
                        'model': model,
                        'prob_val': prob_val.copy(),
                        'pred_val': pred_val.copy()
                    }

            print(
                f"Modelo u1={u1}, a1={a1}, d1={d1}, u2={u2}, a2={a2}, d2={d2} evaluado. "
                f"Mejor actual: recall_1={best['row']['recall_1_sensitivity']:.3f}, "
                f"spec={best['row']['recall_0_specificity']:.3f}, "
                f"threshold={best['row']['threshold']:.2f}, "
                f"score={best['row']['screening_score']:.3f}"
            )

    df_results = pd.DataFrame(results).sort_values(
        ['screening_score', 'recall_1_sensitivity', 'recall_0_specificity'],
        ascending=False
    ).reset_index(drop=True)

    grid_path = os.path.join(output_dir, 'gridsearch_results.csv')
    df_results.to_csv(grid_path, index=False)

    if best is None:
        raise RuntimeError("No se ha podido entrenar/evaluar ningún modelo.")

    best_row = best['row']
    best_model = best['model']
    best_threshold = best_row['threshold']
    best_probs = best['prob_val']
    best_preds = best['pred_val']

    print("\n=== MEJOR MODELO SCREENING PERPETRADOR ===")
    print(pd.Series(best_row))

    # Guardar modelo y metadatos
    best_model_path = os.path.join(output_dir, 'best_model.keras')
    best_model.save(best_model_path)

    best_report = classification_report(
        y_val,
        best_preds,
        output_dict=True,
        zero_division=0
    )

    with open(os.path.join(output_dir, 'best_report.json'), 'w') as f:
        json.dump(best_report, f, indent=2)

    with open(os.path.join(output_dir, 'best_config.json'), 'w') as f:
        json.dump(
            {
                'selection_objective': 'screening: maximize perpetrator recall while retaining minimum specificity/precision',
                'seed': seed,
                'recall_priority_weight': recall_priority_weight,
                'target_recall': target_recall,
                'min_specificity': min_specificity,
                'min_precision': min_precision,
                'best_row': best_row
            },
            f,
            indent=2
        )

    # Guardar conjuntos train/val
    pd.DataFrame(X_train).to_csv(os.path.join(output_dir, 'X_train.csv'), index=False)
    pd.DataFrame(X_val).to_csv(os.path.join(output_dir, 'X_val.csv'), index=False)
    pd.Series(y_train, name='PERPETRADOR').to_csv(os.path.join(output_dir, 'y_train.csv'), index=False)
    pd.Series(y_val, name='PERPETRADOR').to_csv(os.path.join(output_dir, 'y_val.csv'), index=False)

    # Guardar predicciones del MEJOR modelo, no del último.
    df_pred = pd.DataFrame({
        'idx_original': val_original_idx,
        'y_true_perp': y_val.astype(int),
        'y_pred_perp': best_preds.astype(int),
        'y_prob_perp': best_probs.astype(float),
        'threshold_used': float(best_threshold)
    }).sort_values('idx_original').reset_index(drop=True)

    pred_path = os.path.join(output_dir, 'predictions_with_probs.csv')
    df_pred.to_csv(pred_path, index=False)

    print(f"\nSaved grid: {grid_path}")
    print(f"Saved best model: {best_model_path}")
    print(f"Saved predictions: {pred_path}")
    print("\n=== FINAL REPORT — BEST SCREENING MODEL ===")
    print(classification_report(y_val, best_preds, digits=3, zero_division=0))

    return df_results, pd.Series(best_row)

#Montamos el dataset original
feat_df = pd.read_csv('./data/lista_global_vars.csv')
target_df = pd.read_csv('./data/target_col.csv').fillna(0) # fillna imputa los missings con la categoría 0 que es la correcta.
df_merged = feat_df.join(target_df, how='inner')
df_merged = df_merged[~((df_merged["GENERO_BIN_2"] == 1) | (df_merged["ORIENTSEX.BN_3"] == 1))].drop(columns=["GENERO_BIN_2","ORIENTSEX.BN_3"]).reset_index(drop=True)

df_merged_perpretador = df_merged.drop(columns=['VÍCTIMA', 'VICTIMA_PERPETRADOR', 'POLIVICTIMIZACION', 'POLIPERPETRACION', 'SOLO.VICTIMA', 'SOLO.PERPETRADOR', 'NO.VICT_NO.PERP', 'V.O', 'P.SUM.TOTAL', 'V.SUM.TOTAL'])

df_merged_perpetrador_feat = df_merged_perpretador.drop(columns=["PERPETRADOR"])
df_merged_perpetrador_target = df_merged_perpretador["PERPETRADOR"]

#Quitamos el warning de replace
pd.set_option('future.no_silent_downcasting', True)

#Pasamos País a Booleano
df_merged_perpetrador_feat['PAÍS'] = df_merged_perpetrador_feat['PAÍS'].replace({1: True, 2: False})
#Pasamos Etnia a Booleano
df_merged_perpetrador_feat['ETNIA.BN'] = df_merged_perpetrador_feat['ETNIA.BN'].replace({0.0: False, 1.0: True})
#Pasamos Fugas a Booleano
df_merged_perpetrador_feat['FUGAS.BN'] = df_merged_perpetrador_feat['FUGAS.BN'].replace({0.0: False, 1.0: True})
#One-hot encoding de Porno
#df_merged_perpetrador_feat = pd.get_dummies(df_merged_perpetrador_feat, columns=['PORNO.T'])
"""
# ABUSOSUBS2 hereda de ABUSOSUBS1 si es 0
df_merged_perpetrador_feat.loc[df_merged_perpetrador_feat['ABUSOSUBS1'] == 0, 'ABUSOSUBS2'] = 0
# Agrupamos ABUSOSUBS2, categoría False es nunca, anual, mensual True es Diario, Semanal. Dejamos solo la binaria
df_merged_perpetrador_feat['ABUSOSUBS2.BN'] = (~df_merged_perpetrador_feat['ABUSOSUBS2'].isin([1,2,3]))
df_merged_perpetrador_feat = df_merged_perpetrador_feat.drop(columns=["ABUSOSUBS2"])
df_merged_perpetrador_feat = pd.get_dummies(df_merged_perpetrador_feat, columns=['ABUSOSUBS1'])
"""
"""
#Limpiamos genero y orientacion sexual
df_merged_perpetrador_feat = df_merged_perpetrador_feat.drop(columns=["GENERO_BIN_1","ORIENTSEX.BN_2"])
#Renombramos
df_merged_perpetrador_feat = df_merged_perpetrador_feat.rename(columns=
                                                       {'GENERO_BIN_0': 'GENERO.BN', 'ORIENTSEX.BN_1': 'ORIENTSEX.BN'})
#Convertimos a booleano
df_merged_perpetrador_feat['GENERO.BN'] = df_merged_perpetrador_feat['GENERO.BN'].replace({0.0: False, 1.0: True})
df_merged_perpetrador_feat['ORIENTSEX.BN'] = df_merged_perpetrador_feat['ORIENTSEX.BN'].replace({0.0: False, 1.0: True})
"""

#Alternativa, convertimos a booleanos sin limpiar para no perder precisión
df_merged_perpetrador_feat['GENERO.BN0'] = df_merged_perpetrador_feat['GENERO_BIN_0'].replace({0.0: False, 1.0: True})
df_merged_perpetrador_feat['ORIENTSEX.BN0'] = df_merged_perpetrador_feat['ORIENTSEX.BN_1'].replace({0.0: False, 1.0: True})
df_merged_perpetrador_feat['GENERO.BN1'] = df_merged_perpetrador_feat['GENERO_BIN_1'].replace({0.0: False, 1.0: True})
df_merged_perpetrador_feat['ORIENTSEX.BN1'] = df_merged_perpetrador_feat['ORIENTSEX.BN_2'].replace({0.0: False, 1.0: True})
df_merged_perpetrador_feat = df_merged_perpetrador_feat.drop(columns=["GENERO_BIN_0","GENERO_BIN_1","ORIENTSEX.BN_1","ORIENTSEX.BN_2"])

#Convive con hermanos
df_merged_perpetrador_feat = df_merged_perpetrador_feat.rename(columns= {'CONVIVEN.5': 'CONVIVEN_H'})
df_merged_perpetrador_feat['CONVIVEN_H'] = df_merged_perpetrador_feat['CONVIVEN_H'].replace({0.0: False, 1.0: True})
#Conviven con 0 progenitores
df_merged_perpetrador_feat = df_merged_perpetrador_feat.rename(columns= {'CONVIVEN.6': 'CONVIVEN_0'})
df_merged_perpetrador_feat['CONVIVEN_0'] = df_merged_perpetrador_feat['CONVIVEN_0'].replace({0.0: False, 1.0: True})
"""
#Conviven con 1 progenitore
df_merged_perpetrador_feat['CONVIVEN_1'] = ((df_merged_perpetrador_feat[['CONVIVEN.1','CONVIVEN.2','CONVIVEN.3','CONVIVEN.4']] == 1).sum(axis=1) == 1)
#Conviven con 2 progenitores
df_merged_perpetrador_feat['CONVIVEN_2'] = ((df_merged_perpetrador_feat[['CONVIVEN.1','CONVIVEN.2','CONVIVEN.3','CONVIVEN.4']] == 1).sum(axis=1) > 1)
#Limpiamos las conviven viejas
df_merged_perpetrador_feat = df_merged_perpetrador_feat.drop(columns=["CONVIVEN.1","CONVIVEN.2","CONVIVEN.3","CONVIVEN.4"])
"""
#Mostramos el data-set final
display(df_merged_perpetrador_feat)

df_merged_perpetrador_feat.to_csv("df_perpetrador_feat.csv")
df_merged_perpetrador_target.to_csv("df_perpretador_target.csv")

df_pca, df_variance, scaler, pca, means = perform_pca(df_merged_perpetrador_feat)
plot_pca_variance(df_variance)
df_pca_22 = select_PCA_df(df_variance, df_pca, 0.99)
df_variance.to_csv("df_PCA_variance_per.csv")
df_pca_22.to_csv("df_PCA_99.csv")

df_results, best_combination = NN_perpetrator_classifier(
    df_pca_22,
    df_merged_perpetrador_target,
    output_dir='./content/perpetrator',
    batch_size=128,
    grid_params=None,
    tpu_address=None
)

#Mostramos las max_variables variables más importantes de cada PC hasta max_PC
max_PC = 18
max_variables = 5
loadings = pd.DataFrame(
    pca.components_,
    columns=df_merged_perpetrador_feat.columns,
    index=[f"PC{i+1}" for i in range(pca.n_components_)]
)
filas = []
for pc in loadings.index[:max_PC]:
    s = loadings.loc[pc].sort_values(key=abs, ascending=False).head(max_variables)

    fila = {"PC": pc}

    for i, (var, valor) in enumerate(s.items(), start=1):
        fila[f"var_{i}"] = var
        fila[f"loading_{i}"] = valor

    filas.append(fila)
tabla_top_pcs = pd.DataFrame(filas)
display(tabla_top_pcs)

# Comprobación final del CSV de predicciones para SMART AND
data_set = pd.read_csv('./content/perpetrator/predictions_with_probs.csv')

print("=== CHECK predictions_with_probs.csv ===")
print(data_set.head())
print(data_set.shape)
print(data_set.columns.tolist())

true_column = data_set['y_true_perp']
pred_column = data_set['y_pred_perp']

print(classification_report(true_column, pred_column, digits=3))

In [ ]:
# CHECK opcional: ejecutar SOLO después de que haya terminado la celda principal.
# No usa variables internas de la función; lee directamente el CSV generado por el MEJOR modelo.
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from pathlib import Path

pred_path = Path("./content/perpetrator/predictions_with_probs.csv")

if pred_path.exists():
    df_pred = pd.read_csv(pred_path)
    print("Saved:", pred_path)
    print(df_pred.head())
    print(df_pred.shape)
    print(df_pred.columns.tolist())

    if "threshold_used" in df_pred.columns:
        print("\nThreshold usado:", df_pred["threshold_used"].iloc[0])

    print("\n=== FINAL REPORT — PERPETRATOR BEST CSV ===")
    print(classification_report(df_pred["y_true_perp"], df_pred["y_pred_perp"], digits=3, zero_division=0))
    print("\nConfusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(df_pred["y_true_perp"], df_pred["y_pred_perp"], labels=[0, 1]))
else:
    print("Todavía no existe:", pred_path)
    print("Ejecuta primero la celda principal que entrena el modelo.")
